# Análisis

<div style="text-align: justify;">

Este _notebook_ contrasta estadísticamente las 7 variantes del NSGA-II a partir de los CSV por ejecución que genera `Resultados.ipynb` (`Results/Ejecuciones`). Las variantes de la literatura (NSGA2, PLS, MOLS, MOPR) se presentan a la izquierda y las propuestas (DBS, PNS, MPS) a la derecha. Para cada métrica y _dataset_ se usan las 31 semillas como bloques (muestras pareadas, pues todas las variantes comparten las semillas), con **α = 0.05**:

1. **Prueba de Friedman**: contrasta $H_0$: las 7 variantes tienen el mismo rendimiento. Se reporta además el coeficiente de concordancia de Kendall $W = \chi^2_F / (b(k-1))$.
2. **Prueba _post-hoc_ de Wilcoxon** de rangos con signo (pareada, bilateral), solo si se rechaza $H_0$. Las variantes **propuestas son las de control**: cada una se compara contra todas las demás (los pares entre propuestas se evalúan una sola vez; 15 comparaciones por métrica y _dataset_).
3. **Corrección de Holm** sobre esas 15 comparaciones, para controlar el error por familia (FWER).
4. **Tamaño de efecto A12 de Vargha-Delaney**, orientado al control: $A_{12} = P(\text{control mejor}) + 0.5\,P(\text{empate})$. Valores mayores que 0.5 favorecen a la variante de control; magnitud según $|A_{12}-0.5|$: despreciable < 0.06, pequeño < 0.14, mediano < 0.21, grande en otro caso.

Los rangos se asignan de modo que **1 = mejor** (se invierte el sentido en las métricas donde MAYOR es mejor). Se analizan todas las métricas de los resultados: HV, IGD+, Spread, Jaccard, Wang, qscore (máx. y prom.) y % de grupos sin enriquecimiento.

</div>

## Preparación del ambiente

In [1]:
#######################################################
# MODULO DE PREPARACION DEL AMBIENTE Y LECTURA DE RESULTADOS
#######################################################

import os                                                   # Llamadas al SO.
import numpy  as np                                         # Operaciones numericas.
import pandas as pd                                         # Lectura de los CSV por ejecucion.
from pathlib import Path                                    # Formato generalizado de direcciones.
from itertools import combinations
from scipy import stats
from IPython.display import display, Markdown

try:
    import google.colab                                     # Solo disponible dentro de Colab.
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    THESIS_DIR = Path("/content/Thesis")
else:
    _cwd = Path(os.getcwd()).resolve()
    THESIS_DIR = next(
        (p for p in (_cwd, *_cwd.parents) if (p / "Datasets").is_dir() and (p / "Source Code").is_dir()),
        None,
    )
    if THESIS_DIR is None:
        raise FileNotFoundError("No se encontro la carpeta del repositorio (Datasets y Source Code).")

RESULTS_DIR = THESIS_DIR / "Results" / "Ejecuciones"       # CSV generados por Resultados.ipynb.
VARIANTES_LITERATURA = ["NSGA2", "NSGA2_PLS", "NSGA2_MOLS", "NSGA2_MOPR"]   # Variantes de la literatura (izquierda).
VARIANTES_PROPUESTAS = ["NSGA2_DBS", "NSGA2_PNS", "NSGA2_MPS"]                            # Operadores propuestos (derecha).
VARIANTES = VARIANTES_LITERATURA + VARIANTES_PROPUESTAS                      # Orden de todas las tablas.
print(f"[analisis] {RESULTS_DIR}")

[analisis] C:\Users\benja\Desktop\workspace\Thesis\Results\Ejecuciones


## Friedman, post-hoc de Wilcoxon con corrección de Holm y tamaño de efecto A12

In [2]:
#######################################################
# MODULO: FRIEDMAN + POST-HOC DE WILCOXON (HOLM) + TAMANO DE EFECTO A12
#######################################################
# Por (metrica, dataset): bloques = semillas (31), tratamientos = variantes (7).
#   1) Friedman (scipy.stats.friedmanchisquare) + W de Kendall.
#   2) Si Friedman es significativo (p < ALFA): Wilcoxon de rangos con signo
#      (pareado por semilla, bilateral) de cada variante PROPUESTA (control)
#      contra todas las demas. El par propuesta-propuesta se evalua una sola vez.
#      Los p se ajustan por Holm dentro de cada (metrica, dataset).
#   3) Tamano de efecto A12 de Vargha-Delaney, orientado a la variante de control:
#      A12 = P(control mejor que la otra) + 0.5 P(empate). > 0.5 favorece al control.
# Lee los CSV por ejecucion generados en Resultados.ipynb.
#######################################################

ALFA = 0.05

# (archivo, columna, etiqueta, MAYOR es mejor)
METRICAS_ESTADISTICAS = [
    ("igd_plus_ejecuciones.csv",    "hv_final",         "HV",                           True),
    ("igd_plus_ejecuciones.csv",    "igd_plus",         "IGD+",                         False),
    ("spread_ejecuciones.csv",      "spread",           "Spread",                       False),
    ("jaccard_ejecuciones.csv",     "jaccard_promedio", "Jaccard",                      False),
    ("wang_ejecuciones.csv",        "wang_promedio",    "Wang",                         False),
    ("david_ejecuciones.csv",       "qscore_max",       "qscore (max)",                 True),
    ("david_ejecuciones.csv",       "qscore_prom",      "qscore (prom)",                True),
    ("david_ejecuciones.csv",       "pct_sin_enr",      "% grupos sin enriquecimiento", False),
]

def holm(p_valores: np.ndarray) -> np.ndarray:
    """Ajuste de Holm (step-down) de una familia de valores p."""
    p = np.asarray(p_valores, dtype=float)
    m = len(p)
    orden = np.argsort(p)
    ajustado = np.empty(m)
    maximo = 0.0
    for rango, i in enumerate(orden):
        maximo = max(maximo, (m - rango) * p[i])             # Monotonia: el ajustado nunca decrece.
        ajustado[i] = min(1.0, maximo)
    return ajustado

def a12(control: np.ndarray, otra: np.ndarray, mayor_mejor: bool) -> float:
    """A12 de Vargha-Delaney: probabilidad de que `control` sea mejor que `otra` (empates cuentan 0.5)."""
    x, y = np.asarray(control, float), np.asarray(otra, float)
    if not mayor_mejor:
        x, y = -x, -y
    return float(((x[:, None] > y[None, :]).sum() + 0.5 * (x[:, None] == y[None, :]).sum()) / (len(x) * len(y)))

def magnitud_a12(a: float) -> str:
    """Umbrales de Vargha-Delaney sobre |A12 - 0.5|: 0.06 (pequeno), 0.14 (mediano), 0.21 (grande)."""
    d = abs(a - 0.5)
    return "despreciable" if d < 0.06 else "pequeño" if d < 0.14 else "mediano" if d < 0.21 else "grande"

def wilcoxon_pareado(x: np.ndarray, y: np.ndarray) -> tuple:
    """Wilcoxon de rangos con signo bilateral; si todas las diferencias son 0 devuelve p = 1."""
    if np.all(np.asarray(x) == np.asarray(y)):
        return 0.0, 1.0
    r = stats.wilcoxon(x, y, alternative="two-sided")
    return float(r.statistic), float(r.pvalue)

# Pares: cada propuesta contra todas las demas; el par propuesta-propuesta una sola vez.
PARES = []
for ctrl in VARIANTES_PROPUESTAS:
    for otra in VARIANTES:
        if otra == ctrl or (otra in VARIANTES_PROPUESTAS and VARIANTES_PROPUESTAS.index(otra) < VARIANTES_PROPUESTAS.index(ctrl)):
            continue
        PARES.append((ctrl, otra))

filas_friedman, filas_posthoc, rangos_medios = [], [], {}
for archivo, columna, etiqueta, mayor_mejor in METRICAS_ESTADISTICAS:
    ruta = RESULTS_DIR / archivo
    if not ruta.exists():
        print(f"[{etiqueta}] aviso: falta {archivo}, se omite.")
        continue
    df = pd.read_csv(ruta)
    for nombre_ds, df_ds in df.groupby("dataset"):
        tabla = df_ds.pivot(index="semilla", columns="variante", values=columna)
        tabla = tabla[[v for v in VARIANTES if v in tabla.columns]].dropna()
        b, k = tabla.shape
        if k < 3 or b < 2:
            print(f"[{etiqueta} | {nombre_ds}] datos insuficientes (b={b}, k={k}).")
            continue
        chi2, p_f = stats.friedmanchisquare(*[tabla[v].to_numpy() for v in tabla.columns])
        valores = -tabla.to_numpy() if mayor_mejor else tabla.to_numpy()      # 1 = mejor.
        rangos = np.apply_along_axis(stats.rankdata, 1, valores)
        medios = pd.Series(rangos.mean(axis=0), index=tabla.columns)
        rangos_medios[(etiqueta, nombre_ds)] = medios
        significativo = p_f < ALFA
        filas_friedman.append({
            "metrica": etiqueta, "dataset": nombre_ds, "n_bloques": b, "k": k,
            "chi2": chi2, "p_valor": p_f, "W_kendall": chi2 / (b * (k - 1)),
            "significativo": significativo, "mejor_variante": medios.idxmin(),
        })
        pares_ds = [(c, o) for c, o in PARES if c in tabla.columns and o in tabla.columns]
        if significativo:
            res = [wilcoxon_pareado(tabla[c].to_numpy(), tabla[o].to_numpy()) for c, o in pares_ds]
            p_holm = holm(np.array([p for _, p in res]))
        for n, (ctrl, otra) in enumerate(pares_ds):
            a = a12(tabla[ctrl].to_numpy(), tabla[otra].to_numpy(), mayor_mejor)
            fila = {
                "metrica": etiqueta, "dataset": nombre_ds, "control": ctrl, "comparada": otra,
                "rango_control": medios[ctrl], "rango_comparada": medios[otra],
                "mediana_control": tabla[ctrl].median(), "mediana_comparada": tabla[otra].median(),
                "W": np.nan, "p_wilcoxon": np.nan, "p_holm": np.nan, "significativo (Holm)": False,
                "A12": a, "magnitud A12": magnitud_a12(a), "mejor": "n/a (Friedman n.s.)",
            }
            if significativo:
                fila.update({"W": res[n][0], "p_wilcoxon": res[n][1], "p_holm": p_holm[n],
                             "significativo (Holm)": bool(p_holm[n] < ALFA)})
                fila["mejor"] = (ctrl if a > 0.5 else otra) if p_holm[n] < ALFA else "-"
            filas_posthoc.append(fila)

friedman = pd.DataFrame(filas_friedman)
posthoc  = pd.DataFrame(filas_posthoc)
friedman.to_csv(RESULTS_DIR / "friedman_resultados.csv", index=False)
posthoc.to_csv(RESULTS_DIR / "posthoc_wilcoxon_holm.csv", index=False)
print(f"alfa = {ALFA} | {len(friedman)} pruebas de Friedman | {len(posthoc)} comparaciones control vs. resto "
      f"({len(PARES)} por metrica y dataset; guardadas en {RESULTS_DIR})")

alfa = 0.05 | 27 pruebas de Friedman | 405 comparaciones control vs. resto (15 por metrica y dataset; guardadas en C:\Users\benja\Desktop\workspace\Thesis\Results\Ejecuciones)


## Tabla por métrica

Una tabla por métrica con una fila por _dataset_ y par (control propuesto, variante comparada): rangos medios, medianas, p de Friedman, p de Wilcoxon y de Holm, A12 y su magnitud, y la variante mejor. Colores de la variante ganadora: verde = gana el control (propuesta), azul = gana la variante comparada, amarillo = sin diferencia significativa (Holm); sin color cuando Friedman no es significativo.

In [3]:
#######################################################
# MODULO: UNA TABLA POR METRICA
#######################################################
# Una fila por (dataset, variante de control, variante comparada) con rangos medios
# (1 = mejor), medianas, Friedman del dataset, Wilcoxon, p de Holm, A12 y la
# variante que resulta mejor. Colores: verde = gana el control (diferencia significativa
# con Holm), azul = gana la variante comparada, amarillo = sin diferencia significativa.
#######################################################

COLUMNAS_TABLA = ["Friedman p", "rango_control", "rango_comparada", "mediana_control", "mediana_comparada",
                  "p_wilcoxon", "p_holm", "A12", "magnitud A12", "mejor", "significativo (Holm)"]

COLOR_GANA, COLOR_EMPATE, COLOR_PIERDE = "#b7e1b0", "#fff2a8", "#a9cdf0"     # verde, amarillo, azul

def colorear_mejor(fila: pd.Series) -> list:
    """Verde: gana el control; azul: gana la variante comparada; amarillo: sin diferencia (Holm). Sin color si Friedman n.s."""
    if fila["mejor"].startswith("n/a"):
        return [""] * len(fila)
    if not fila["significativo (Holm)"]:
        color = COLOR_EMPATE
    else:
        color = COLOR_GANA if fila["mejor"] == fila.name[1] else COLOR_PIERDE    # name = (dataset, control, comparada)
    return [f"background-color: {color}" if c in ("p_holm", "A12", "magnitud A12", "mejor") else "" for c in fila.index]

TABLAS_METRICAS = {}
for _, columna, etiqueta, mayor_mejor in METRICAS_ESTADISTICAS:
    sub = posthoc[posthoc["metrica"] == etiqueta].merge(
        friedman[["metrica", "dataset", "p_valor"]].rename(columns={"p_valor": "Friedman p"}), on=["metrica", "dataset"])
    if sub.empty:
        continue
    sub = sub.set_index(["dataset", "control", "comparada"])[COLUMNAS_TABLA]
    TABLAS_METRICAS[etiqueta] = sub
    sentido = "MAYOR es mejor" if mayor_mejor else "MENOR es mejor"
    display(Markdown(f"### {etiqueta} ({sentido}; α = {ALFA})"))
    display(sub.style.format({"Friedman p": "{:.2e}", "rango_control": "{:.2f}", "rango_comparada": "{:.2f}",
                              "mediana_control": "{:.4g}", "mediana_comparada": "{:.4g}",
                              "p_wilcoxon": "{:.2e}", "p_holm": "{:.2e}", "A12": "{:.3f}"}, na_rep="-")
            .apply(colorear_mejor, axis=1).hide(axis="columns", subset=["significativo (Holm)"]))
    sub.to_csv(RESULTS_DIR / f"tabla_{columna}.csv", encoding="utf-8-sig")

### HV (MAYOR es mejor; α = 0.05)

### IGD+ (MENOR es mejor; α = 0.05)

### Spread (MENOR es mejor; α = 0.05)

### Jaccard (MENOR es mejor; α = 0.05)

### Wang (max) (MAYOR es mejor; α = 0.05)

### Wang (prom) (MAYOR es mejor; α = 0.05)

### qscore (max) (MAYOR es mejor; α = 0.05)

### qscore (prom) (MAYOR es mejor; α = 0.05)

### % grupos sin enriquecimiento (MENOR es mejor; α = 0.05)

## Resumen global

Victorias, empates y derrotas significativas (Holm) de cada propuesta frente a las variantes de la literatura, por métrica y _dataset_.

In [4]:
#######################################################
# MODULO: RESUMEN GLOBAL (VICTORIAS DE CADA PROPUESTA POR METRICA)
#######################################################
# Para cada variante propuesta, metrica y dataset: contra cuantas variantes gana,
# empata (sin diferencia significativa) o pierde con significancia (Holm, alfa).
# Solo se cuentan las comparaciones contra variantes de la literatura.
#######################################################

lit = posthoc[posthoc["comparada"].isin(VARIANTES_LITERATURA)].copy()
lit["resultado"] = np.select(
    [lit["mejor"] == lit["control"], lit["mejor"] == lit["comparada"], lit["mejor"] == "-"],
    ["gana", "pierde", "empata"], default="n/a (Friedman n.s.)")
resumen_global = (lit.groupby(["metrica", "dataset", "control"])["resultado"].value_counts().unstack(fill_value=0))
for col in ("gana", "empata", "pierde", "n/a (Friedman n.s.)"):
    if col not in resumen_global:
        resumen_global[col] = 0
resumen_global = resumen_global[["gana", "empata", "pierde", "n/a (Friedman n.s.)"]]
resumen_global = resumen_global.reindex([e for _, _, e, _ in METRICAS_ESTADISTICAS], level="metrica")
resumen_global.to_csv(RESULTS_DIR / "resumen_propuestas_vs_literatura.csv", encoding="utf-8-sig")
display(Markdown(f"Cada fila suma {len(VARIANTES_LITERATURA)} comparaciones (propuesta vs. cada variante de la literatura)."))
display(resumen_global)

Cada fila suma 4 comparaciones (propuesta vs. cada variante de la literatura).

resultado                                       gana  empata  pierde  \
metrica                      dataset control                           
HV                           B1      NSGA2_DBS     3       1       0   
                                     NSGA2_PNS     3       1       0   
                             B2      NSGA2_DBS     4       0       0   
                                     NSGA2_PNS     3       1       0   
                             B3      NSGA2_DBS     3       1       0   
                                     NSGA2_PNS     4       0       0   
IGD+                         B1      NSGA2_DBS     4       0       0   
                                     NSGA2_PNS     4       0       0   
                             B2      NSGA2_DBS     3       1       0   
                                     NSGA2_PNS     3       1       0   
                             B3      NSGA2_DBS     3       1       0   
                                     NSGA2_PNS     3       1       0   
Spread                       B1      NSGA2_DBS     0       0       0   
                                     NSGA2_PNS     0       0       0   
                             B2      NSGA2_DBS     0       0       0   
                                     NSGA2_PNS     0       0       0   
                             B3      NSGA2_DBS     0       0       0   
                                     NSGA2_PNS     0       0       0   
Jaccard                      B1      NSGA2_DBS     0       3       1   
                                     NSGA2_PNS     0       3       1   
                             B2      NSGA2_DBS     0       0       0   
                                     NSGA2_PNS     0       0       0   
                             B3      NSGA2_DBS     0       3       1   
                                     NSGA2_PNS     0       3       1   
Wang (max)                   B1      NSGA2_DBS     0       1       3   
                                     NSGA2_PNS     0       1       3   
                             B2      NSGA2_DBS     0       4       0   
                                     NSGA2_PNS     0       4       0   
                             B3      NSGA2_DBS     0       4       0   
                                     NSGA2_PNS     0       4       0   
Wang (prom)                  B1      NSGA2_DBS     1       3       0   
                                     NSGA2_PNS     2       2       0   
                             B2      NSGA2_DBS     0       4       0   
                                     NSGA2_PNS     0       4       0   
                             B3      NSGA2_DBS     0       4       0   
                                     NSGA2_PNS     0       4       0   
qscore (max)                 B1      NSGA2_DBS     1       3       0   
                                     NSGA2_PNS     1       3       0   
                             B2      NSGA2_DBS     4       0       0   
                                     NSGA2_PNS     1       3       0   
                             B3      NSGA2_DBS     0       0       0   
                                     NSGA2_PNS     0       0       0   
qscore (prom)                B1      NSGA2_DBS     4       0       0   
                                     NSGA2_PNS     4       0       0   
                             B2      NSGA2_DBS     0       0       4   
                                     NSGA2_PNS     0       3       1   
                             B3      NSGA2_DBS     1       3       0   
                                     NSGA2_PNS     0       4       0   
% grupos sin enriquecimiento B1      NSGA2_DBS     0       0       0   
                                     NSGA2_PNS     0       0       0   
                             B2      NSGA2_DBS     0       3       1   
                                     NSGA2_PNS     1       3       0   
                             B3      NSGA2_DBS     0       4       0   
                                     NSG